# Introduction to Qrisp
Qrisp is an open-source project developed across organizations. Main features:
1) Typed quantum variables: Qrisp algorithms are constituted of variables and functions instead of qubits and circuits.
2) Modularity: Automated qubit allocation allows separate modules to recycle qubit resources for each other without intertwining the code.
3) Memory management: QuantumVariables can be uncomputed automatically once they are not needed anymore. (Uncomputation could be a potential seminar presentation!)
4) Many High level implementations of State of the Art Algorithms 
5) high-performance simulator
6) Jax integration: Exert hybrid real-time algorithm control with Catalyst and Jasp. Qrisp’s JAX integration enables seamless hybrid quantum-classical workflows with deep integration into classical compiler infrastructure (LLVM/MLIR). 

This tutorial is inspired by:
https://qrisp.eu/general/tutorial/tutorial.html

Many other tutorials: https://qrisp.eu/general/tutorial/index.html & https://qrisp.eu/reference/Examples/index.html

## First examples

In [ ]:
from qrisp import h, cx
from qrisp import QuantumVariable

qv = QuantumVariable(2)
h(qv[0])
cx(qv[0],qv[1])

print(qv.qs)

In [ ]:
qv.get_measurement()

In [ ]:
qv.qs.statevector()

A cool thing about qrisp is that we can do the hadamard transform as follows without a for loop as we did with Pennylane:

In [ ]:
n_qubits=5
qv = QuantumVariable(n_qubits)
h(qv)
print(qv.qs)

## Quantum Floats

Let's look at a special subclass of the QuantumVariable: QuantumFloats!

`msize` indicates the amount of mantissa qubits, which represent the significant digits of a number in a floating-point representation


$$
x_i = i \cdot 2^{e}, \qquad
i \in
\begin{cases}
\{0, 1, \dots, 2^{n}-1\} & \text{unsigned } (n \text{ qubits}) \\
\{-2^{n}, \dots, 2^{n}-1\} & \text{signed } (n+1 \text{ qubits})
\end{cases}
$$

where $n$ is the number of mantissa qubits (`msize`) and $e$ the exponent.

In [ ]:
from qrisp import QuantumFloat


a = QuantumFloat(msize=3, exponent=-2, signed=False)
h(a)
print(a)

With `QuantumFloat(n, -n)` the register represents $x = i/2^n \in [0,1)$, i.e. the binary fraction $0.b_1 b_2 \dots b_n$, which is exactly the format of the QPE phase $\varphi$.

In [ ]:
precision=3
a = QuantumFloat(precision, -precision)
h(a)
print(a)


In [ ]:
for i in range(2**a.size): print(i,i*2**(-precision),a.decoder(i))

Let's see some Quantum Arithmetics

In [ ]:
a = QuantumFloat(3)
b = QuantumFloat(3)

a[:] = 5
b[:] = 2

c = a + b
d = a - c
e = d * b
f = a / b

In [ ]:
compiled=c.qs.compile()

In [ ]:
compiled.count_ops()

In [ ]:
compiled.depth(),compiled.t_depth()

## QPE example

Let's go straight to QPE, you will see how can be implemented with less effort than pennylane.

We use `with control(res[i])` because allows for controlled unitaries! This statement enters a ControlEnvironment such that every quantum operation inside the indented code block will be controlled on the i-th qubit of res (https://qrisp.eu/reference/Quantum%20Environments/ControlEnvironment.html)


In [ ]:
from qrisp import QuantumFloat, control, QFT, h


def QPE(psi, U, precision):
    res = QuantumFloat(precision, -precision)
    h(res)
    for i in range(precision):
        with control(res[i]):
            for _ in range(2**i):
                U(psi)
    return QFT(res, inv=True) # inv=True does what qp.adjoint does in Pennylane

"with control" allows us to run controlled unitaries without explicitly implement them as we did with Pennylane

In [ ]:
from qrisp import p, QuantumVariable
import numpy as np


def U(psi):
    phi_1 = 0.5
    phi_2 = 0.125

    p(phi_1 * 2 * np.pi, psi[0])
    p(phi_2 * 2 * np.pi, psi[1])

psi = QuantumVariable(2)

h(psi)

res = QPE(psi, U, 3)

In [ ]:
res.qs.statevector()

Let's compare with the bult-in result:

In [ ]:
from qrisp import QPE


psi = QuantumVariable(2)

h(psi)
res = QPE(psi, U, 3)
res.qs.statevector()

## Grover

In the pennylane version, the bitstrings "combos" were written in by hand, but for real problems we don't know the solutions in advance obviously. Let's have an example: we want to find all the $x$ within $[-4,3.5]$ s.t. $x^2=0.25$, assuming that we don't know the solutions

In [ ]:
from qrisp import QuantumFloat, auto_uncompute, z
from qrisp.grover import grovers_alg


def sqrt_oracle(x):
    ### TODO ###

qf = QuantumFloat( )


qf*qf == 0.25 checks every x and writes the result into one extra qubit: 1 for "right answer", 0 for "wrong answer".
z is the gate that adds a minus sign whenever that qubit is 1. So only the right answers get the minus sign.


In [ ]:
grovers_alg(qf, sqrt_oracle)
print(qf)

let's see what "uncompute" does

In [ ]:
sqrt_oracle(qf)
print(qf.qs)

and have a look at the resources:

In [ ]:
compiled=qf.qs.compile()
compiled.count_ops(),compiled.depth(),compiled.t_depth()